<a href="https://colab.research.google.com/github/DanyaLeyva/scottish-gaelic-translation-evaluation/blob/main/_ScottishGaelic_to_English_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM-Augmented Neural Machine Translation Evaluation

### Project Overview
This project evaluates whether Large Language Models (LLMs) can improve Neural Machine Translation (NMT) outputs, especially for low-resource languages.

### Goals:
- Compare baseline NMT vs improved outputs
- Use consistent evaluation metrics:
  - BLEU
  - chrF
  - COMET
- Apply LLM post-editing to improve translations
- Analyze performance differences

### Language Pairs:
- High-resource: [English]
- Low-resource: [Scottish, Gaelic]

# ============================================
# IMPORTANT: Install libraries at the TOP only
# ============================================
# Installing mid-notebook can crash Colab and reset training.
# This causes loss of epoch progress.
# Always install once at the beginning and restart runtime.

In [ ]:
# Install sentence-transformers for LaBSE filtering 4
!pip install -q sentence-transformers

In [ ]:

# Install evaluation libraries 8
!pip uninstall -y tensorflow tf-keras keras >/dev/null 2>&1 || true
!pip install -q --upgrade sacrebleu unbabel-comet

In [ ]:
# Install libraries needed for fine-tuning12
!pip install -q datasets accelerate sentencepiece

## 1. Upload Data Files

This section uploads the parallel corpus files used in the project.
Each time the notebook is restarted, the dataset files must be uploaded again unless they are stored in Google Drive.

Expected files:
- `CCMatrix.en-gd.en` → English sentences
- `CCMatrix.en-gd.gd` → Scottish Gaelic sentences

In [ ]:
import pandas as pd
en_file = "CCMatrix.en-gd.en"
gd_file ="CCMatrix.en-gd.gd"

with open(en_file, encoding = "utf-8") as f: en_sentences = [line.strip() for line in f]
with open(gd_file, encoding = "utf-8") as f: gd_sentences = [line.strip() for line in f]

print("English sentences: ", len(en_sentences))
print("Gaelic sentences: ", len(gd_sentences))

print("\nExamples:")
print(en_sentences[0])
print(gd_sentences[0])

## 2. Confirm Dataset Integrity

This section verifies that the English and Scottish Gaelic datasets are properly aligned.

If a mismatch in sentence counts is detected, both datasets are trimmed to the same minimum length.  
This ensures that each English sentence correctly corresponds to a Gaelic sentence for translation tasks.

In [ ]:
# ============================================
# 2. Confirm Dataset Integrity
# ============================================

# This section ensures that both the English and Gaelic datasets
# contain the same number of sentence pairs.

# If a mismatch is detected, both datasets will be trimmed
# to the same minimum length to preserve alignment.

# Check whether the English and Gaelic files have the same number of lines
if len(en_sentences) != len(gd_sentences):

    # Print a warning if the counts do not match
    print("⚠️ Mismatch detected between files")
    print("English sentence count:", len(en_sentences))
    print("Gaelic sentence count:", len(gd_sentences))

    # Find the minimum length so both lists can be trimmed equally
    min_len = min(len(en_sentences), len(gd_sentences))

    # Trim both sentence lists to preserve pair alignment
    en_sentences = en_sentences[:min_len]
    gd_sentences = gd_sentences[:min_len]

    print("Datasets trimmed to:", min_len)

else:
    # Confirm that the datasets are already aligned
    print("Dataset integrity check passed.")

# Print final aligned sentence count
print("Total aligned sentence pairs:", len(en_sentences))

## 3. Create Structured Dataset

This section combines the English and Scottish Gaelic sentences into a structured format for easier processing and evaluation.

Each row represents a parallel sentence pair.

In [ ]:
# Create a DataFrame of aligned sentence pairs
data = pd.DataFrame({
    "english": en_sentences,
    "gaelic": gd_sentences
})

# Display basic info
print("Dataset shape:", data.shape)
data.head()

In [ ]:
# Save a copy of the raw dataset before cleaning
data_before_cleaning = data.copy()

## 4. Data Cleaning

This section removes empty, corrupted, or invalid sentence pairs while preserving English–Scottish Gaelic alignment.

Cleaning is performed on sentence pairs directly so that source and target sentences remain correctly matched.

In [ ]:
import re

# Detect corrupted text (replacement characters often caused by encoding issues)
def is_corrupted(text):
    return "�" in str(text)

# Detect unexpected non-Latin scripts such as Japanese, Cyrillic, Arabic, or Hebrew
unexpected_script_pattern = re.compile(
    r'[\u3040-\u30ff\u4e00-\u9faf\u0400-\u04FF\u0600-\u06FF\u0590-\u05FF]'
)

def has_unexpected_script(text):
    return bool(unexpected_script_pattern.search(str(text)))

In [ ]:
# Clean aligned sentence pairs while preserving English-Gaelic matching
clean_pairs = []

for en, gd in zip(en_sentences, gd_sentences):
    en = en.strip()
    gd = gd.strip()

    # Skip empty sentence pairs
    if not en or not gd:
        continue

    # Skip very short sentence pairs
    if len(en) <= 3 or len(gd) <= 3:
        continue

    # Skip corrupted text
    if is_corrupted(en) or is_corrupted(gd):
        continue

    # Skip rows containing unexpected scripts
    if has_unexpected_script(en) or has_unexpected_script(gd):
        continue

    clean_pairs.append((en, gd))

print("Clean aligned pairs:", len(clean_pairs))

In [ ]:
# Rebuild the cleaned dataset from valid aligned pairs
data = pd.DataFrame(clean_pairs, columns=["english", "gaelic"])

print("Cleaned dataset size:", len(data))
data.head()

In [ ]:
# Save a copy of the cleaned dataset
data_after_cleaning = data.copy()

In [ ]:
# Compare dataset size before and after cleaning
print("Before cleaning:", len(data_before_cleaning))
print("After cleaning:", len(data_after_cleaning))
print("Rows removed:", len(data_before_cleaning) - len(data_after_cleaning))

## 4.2 Improved Alignment Filtering

To address misalignment, additional filtering rules were introduced to remove sentence pairs that are likely unrelated or noisy.

These include:
- Removing unusually long mismatches
- Removing template or metadata text
- Filtering out non-natural language content

In [ ]:
# ============================================
# 4.2 Improved Alignment Filtering
# ============================================

# This section applies stricter filtering rules
# to remove sentence pairs that are likely noisy,
# weakly aligned, repetitive, or non-natural.

import re

def has_too_many_repeated_words(text, threshold=4):
    """
    Return True if the same word repeats too many times.
    Example: 'aonachd aonachd aonachd aonachd'
    """
    words = text.lower().split()
    if not words:
        return False

    counts = {}
    for word in words:
        counts[word] = counts.get(word, 0) + 1

    return max(counts.values()) >= threshold


def has_repeated_characters(text, threshold=8):
    """
    Return True if the text contains a character repeated many times.
    Example: 'dddddddddddd'
    """
    return re.search(rf"(.)\1{{{threshold},}}", text.lower()) is not None


def has_weird_artifacts(text):
    """
    Return True if the text contains common web/template/junk patterns.
    """
    text_lower = text.lower()

    junk_patterns = [
        "template",
        "dàta brataich",
        "link:",
        "http",
        "www",
        "web",
        "ico",
        "download",
        "downloads"
    ]

    return any(pattern in text_lower for pattern in junk_patterns)


def has_too_many_symbols(text):
    """
    Return True if the text contains too many non-letter symbols.
    """
    symbol_count = sum(1 for ch in text if not ch.isalnum() and not ch.isspace())
    return symbol_count > max(10, len(text) * 0.2)


def looks_like_reference_noise(text):
    """
    Return True if the text looks like citation/reference fragments.
    Example: '[2]' or '[5]' or many bracketed numbers.
    """
    bracket_refs = len(re.findall(r"\[\d+\]", text))
    return bracket_refs >= 1


def has_too_many_uppercase_words(text):
    """
    Return True if the line contains many ALL-CAPS words,
    which often indicates titles, headers, or noisy metadata.
    """
    words = text.split()
    uppercase_words = [w for w in words if len(w) > 2 and w.isupper()]
    return len(uppercase_words) >= 4


def is_reasonably_aligned(en, gd):
    # Normalize text for safer comparisons
    en_lower = en.lower().strip()
    gd_lower = gd.lower().strip()

    # Remove extreme length differences
    if abs(len(en) - len(gd)) > 60:
        return False

    # Remove very short or very long lines
    if len(en) < 8 or len(gd) < 8:
        return False
    if len(en) > 250 or len(gd) > 250:
        return False

    # Remove metadata / template / web junk
    if has_weird_artifacts(en) or has_weird_artifacts(gd):
        return False

    # Remove highly repetitive text
    if has_too_many_repeated_words(en) or has_too_many_repeated_words(gd):
        return False

    # Remove repeated-character junk
    if has_repeated_characters(en) or has_repeated_characters(gd):
        return False

    # Remove symbol-heavy junk
    if has_too_many_symbols(en) or has_too_many_symbols(gd):
        return False

    # Remove citation/reference noise
    if looks_like_reference_noise(en) or looks_like_reference_noise(gd):
        return False

    # Remove header/title style noise
    if has_too_many_uppercase_words(en) or has_too_many_uppercase_words(gd):
        return False

    return True


# Apply stricter filtering to the aligned sentence pairs
clean_pairs = [
    (en, gd) for en, gd in clean_pairs
    if is_reasonably_aligned(en, gd)
]

print("After stricter alignment filtering:", len(clean_pairs))

In [ ]:
# Rebuild the dataset after improved alignment filtering
data = pd.DataFrame(clean_pairs, columns=["english", "gaelic"])

print("Dataset size after improved alignment filtering:", len(data))
data.head()

### 4.3 Semantic Trust Filtering with LaBSE

This section applies semantic filtering using LaBSE sentence embeddings.

Unlike earlier rule-based cleaning steps, this method estimates whether each English–Scottish Gaelic sentence pair is semantically similar. Pairs with low semantic similarity are removed.

This helps reduce weakly aligned or unrelated translations that remain after surface-level cleaning.

In [ ]:
# ============================================
# 4.3 Semantic Trust Filtering with LaBSE
# ============================================

from sentence_transformers import SentenceTransformer, util
import torch

In [ ]:
# Set device (GPU if available, otherwise CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

In [ ]:
labse_model = SentenceTransformer(
    "sentence-transformers/LaBSE",
    device=device
)

print("LaBSE model loaded.")

In [ ]:
# ============================================
# 4.3.1 Compute semantic similarity for sentence pairs
# ============================================

# Create lists from the current cleaned dataframe
english_list = data["english"].tolist()
gaelic_list = data["gaelic"].tolist()

# Encode English and Gaelic sentences
english_embeddings = labse_model.encode(
    english_list,
    convert_to_tensor=True,
    show_progress_bar=True,
    batch_size=64
)

gaelic_embeddings = labse_model.encode(
    gaelic_list,
    convert_to_tensor=True,
    show_progress_bar=True,
    batch_size=64
)

# Compute cosine similarity for each aligned pair
similarities = util.cos_sim(english_embeddings, gaelic_embeddings).diagonal()

print("Similarity computation complete.")
print("Number of pairs scored:", len(similarities))

### 4.3.2 Similarity Threshold

A similarity threshold is used to keep only higher-confidence sentence pairs.

This threshold can be adjusted after inspecting the score distribution. A higher threshold keeps fewer but more reliable pairs.

In [ ]:
# ============================================
# 4.3.2 Inspect similarity scores
# ============================================

# Convert tensor to Python list
similarity_scores = similarities.cpu().tolist()

print("Minimum similarity:", min(similarity_scores))
print("Maximum similarity:", max(similarity_scores))
print("Average similarity:", sum(similarity_scores) / len(similarity_scores))

In [ ]:
# ============================================
# 4.3.3 Filter low-similarity pairs
# ============================================

similarity_threshold = 0.60

# Keep only sentence pairs above threshold
trusted_pairs = []

for en, gd, score in zip(english_list, gaelic_list, similarity_scores):
    if score >= similarity_threshold:
        trusted_pairs.append((en, gd, score))

print("Trusted pairs kept:", len(trusted_pairs))
print("Pairs removed:", len(english_list) - len(trusted_pairs))

In [ ]:
# ============================================
# 4.3.4 Rebuild dataset after LaBSE filtering
# ============================================

data = pd.DataFrame(
    [(en, gd) for en, gd, score in trusted_pairs],
    columns=["english", "gaelic"]
)

print("Dataset size after LaBSE filtering:", len(data))
data.head()

### 4.3.5 LaBSE Filtering Analysis

LaBSE filtering removes sentence pairs that are less likely to match semantically.

This step is intended to improve dataset quality beyond rule-based cleaning by keeping only higher-confidence translation pairs. The filtered dataset will be used for the remaining translation and evaluation pipeline.

In [ ]:
# ============================================
# 4.3.5 Show sample trusted pairs
# ============================================

for i in range(5):
    print(f"\nExample {i+1}")
    print("English:", data.iloc[i]["english"])
    print("Gaelic:", data.iloc[i]["gaelic"])
    print("-" * 60)

### 4.4 Post-LaBSE Data Refinement

After LaBSE filtering, some sentence pairs still appear noisy, overly short, title-like, or only partially aligned.

This section applies an additional cleanup pass to remove:
- Very short fragments
- All-uppercase or title-like text
- Low-information pairs
- Remaining noisy pairs that are unlikely to help translation training

This refinement is intended to improve the final dataset quality before splitting and evaluation.

In [ ]:
# ============================================
# 4.4 Post-LaBSE Data Refinement
# ============================================

import re

def is_mostly_uppercase(text):
    """
    Returns True if most alphabetic characters are uppercase.
    Helps remove title-like or shouty metadata text.
    """
    letters = [ch for ch in text if ch.isalpha()]
    if len(letters) == 0:
        return False
    upper_ratio = sum(ch.isupper() for ch in letters) / len(letters)
    return upper_ratio > 0.6

def is_too_short(text, min_words=3):
    """
    Remove very short fragments that usually do not provide enough translation signal.
    """
    return len(text.split()) < min_words

def looks_like_title_or_label(text):
    """
    Detect short title-like strings, labels, or headline fragments.
    """
    words = text.split()

    # Very short title-like phrase
    if len(words) <= 4 and text.istitle():
        return True

    # All caps label-like phrase
    if is_mostly_uppercase(text) and len(words) <= 6:
        return True

    return False

def has_too_many_symbols(text):
    """
    Remove rows with too many non-letter symbols.
    """
    if len(text) == 0:
        return True
    symbol_count = sum(1 for ch in text if not ch.isalnum() and not ch.isspace())
    return symbol_count / max(len(text), 1) > 0.20

def looks_low_information(text):
    """
    Remove rows that are too generic or low-information.
    """
    generic_phrases = [
        "world cup",
        "last year's quarter",
        "review by",
        "by china",
        "splash proof",
    ]

    text_lower = text.lower()
    return any(phrase in text_lower for phrase in generic_phrases)

In [ ]:
# ============================================
# 4.4.2 Apply extra refinement rules
# ============================================

before_refinement = len(data)

data = data[
    ~data["english"].apply(is_too_short) &
    ~data["gaelic"].apply(is_too_short) &
    ~data["english"].apply(looks_like_title_or_label) &
    ~data["gaelic"].apply(looks_like_title_or_label) &
    ~data["english"].apply(has_too_many_symbols) &
    ~data["gaelic"].apply(has_too_many_symbols) &
    ~data["english"].apply(looks_low_information) &
    ~data["gaelic"].apply(looks_low_information)
].reset_index(drop=True)

after_refinement = len(data)

print("Before post-LaBSE refinement:", before_refinement)
print("After post-LaBSE refinement:", after_refinement)
print("Rows removed:", before_refinement - after_refinement)

In [ ]:
# ============================================
# 4.4.3 Show sample refined pairs
# ============================================

for i in range(10):
    print(f"\nExample {i+1}")
    print("English:", data.iloc[i]["english"])
    print("Gaelic:", data.iloc[i]["gaelic"])
    print("-" * 60)

### 4.4.4 Refinement Analysis

This additional refinement step removes low-information and title-like sentence pairs that remained after LaBSE filtering.

The goal is not only to improve semantic alignment, but also to ensure that the final dataset contains training examples that are informative enough for machine translation.

### 4.5 Manual Audit of Remaining Sentence Pairs

A small manual audit was conducted to estimate how many sentence pairs remain:
- Well aligned
- Partially aligned
- Clearly misaligned or noisy

This provides a more honest measure of dataset quality beyond automatic filtering alone.

In [ ]:
# ============================================
# 4.5 Manual Audit Sample
# ============================================

audit_sample = data.sample(n=20, random_state=42).reset_index(drop=True)

for i in range(len(audit_sample)):
    print(f"\nAudit Example {i+1}")
    print("English:", audit_sample.iloc[i]["english"])
    print("Gaelic:", audit_sample.iloc[i]["gaelic"])
    print("-" * 70)

### 4.5.1 Manual Audit Summary

A manual audit of 20 randomly selected sentence pairs was conducted.

Results:
- Well aligned: 12 / 20
- Partially aligned: 5 / 20
- Misaligned or noisy: 3 / 20

This confirms that although filtering significantly improved the dataset, some weak or incomplete translation pairs still remain.

## 5. Train/Test Split (Fixed Test Set)

This section creates a fixed test set that will be used consistently across all experiments.

Using the same test set for BLEU, chrF, and COMET ensures that score differences come from model changes, not from changes in the data split.

In [ ]:
# Shuffle once for reproducibility
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Create train, validation, and test splits
train_end = int(0.8 * len(data))
val_end = int(0.9 * len(data))

train_data = data.iloc[:train_end].copy()
val_data = data.iloc[train_end:val_end].copy()
test_data = data.iloc[val_end:].copy()

print("Train size:", len(train_data))
print("Validation size:", len(val_data))
print("Test size:", len(test_data))

In [ ]:
# IMPORTANT:
# Keep the test set fixed for all future evaluations.
# Do not reshuffle or redefine test_data after this point.

test_source = test_data["english"].tolist()
test_reference = test_data["gaelic"].tolist()

print("Example test source:", test_source[0])
print("Example test reference:", test_reference[0])

## 6. Save Fixed Test Set

This section saves the cleaned train, validation, and test splits so the same dataset partitions can be reused consistently across all experiments.

Saving these files improves reproducibility and ensures that BLEU, chrF, and COMET evaluations are consistent.

Below, we display example sentence pairs from the fixed test set to confirm correct English–Scottish Gaelic alignment.

In [ ]:
# Save the cleaned train, validation, and test splits
train_data.to_csv("train_data.csv", index=False)
val_data.to_csv("val_data.csv", index=False)
test_data.to_csv("test_data.csv", index=False)

print("Saved train_data.csv, val_data.csv, and test_data.csv")

In [ ]:
import os

print("train_data.csv exists:", os.path.exists("train_data.csv"))
print("val_data.csv exists:", os.path.exists("val_data.csv"))
print("test_data.csv exists:", os.path.exists("test_data.csv"))

In [ ]:
# Show example test pairs to verify alignment
print("Example test sentence pairs:\n")

for i in range(3):
    print(f"Example {i+1}")
    print("English:", test_source[i])
    print("Gaelic:", test_reference[i])
    print("-" * 50)

### Dataset Limitation: Misaligned Sentence Pairs

During evaluation, some English–Gaelic sentence pairs were found to be misaligned.
Examples include sentences where the meanings do not correspond or contain unrelated content.

This issue likely originates from noise in the CCMatrix dataset, which is automatically
constructed from web data.

While cleaning removed corrupted and non-Gaelic text, semantic misalignment remains a limitation.

This highlights a key challenge in low-resource machine translation:
data quality can significantly impact model performance.

Despite applying multiple cleaning and filtering steps, semantic misalignment persists in some sentence pairs. This reflects a broader challenge in low-resource machine translation, where automatically collected corpora may contain noisy or weakly aligned data.

### Dataset Limitation: Remaining Misalignment

During evaluation, some English–Gaelic sentence pairs were still found to be weakly aligned or incomplete. Although the stricter filtering steps removed much of the obvious noise, some sentence pairs remain only partially aligned due to the nature of the CCMatrix corpus, which is automatically collected from web data.

This means that while dataset quality improved significantly after cleaning, semantic alignment is still not guaranteed for every example. This limitation should be considered when interpreting BLEU, chrF, and COMET scores.

#### Example sentence pairs with reference translations

Example 1  
English (dataset): The impact was that they would report a [...]  
Gaelic (dataset): Tha na làithean a tha eiseachd a [...]  
English (reference translation from Gaelic - Translate.com): The days are heard  

This demonstrates a clear semantic mismatch between the English sentence and its paired Gaelic translation.

---

Example 2  
English (dataset): 2019, August 12 | by Kimbat Isakova  
Gaelic (dataset): An Dùbhlachd 2016 aig aois 69 an dèidh sabaid le aillse.  
English (reference translation from Gaelic - Translate.com/Lingvanex): December 2016 at the age of 69 after a fight with cancer.  

This example shows that the paired sentences are completely unrelated in meaning.

---

Example 3  
English (dataset): A guesstimate is 10,000.  
Gaelic (dataset): Thasing loghdachadh de 10,000.  
English (reference translation from Gaelic - Translate.com/Lingvanex): Thasing reduction of 10,000.  

This demonstrates that the dataset translation does not accurately reflect the meaning of the original English sentence.



## 7. Baseline Translation (NLLB)

This section generates baseline translations using a pre-trained NLLB model.

The model translates English sentences into Scottish Gaelic using the fixed test set.  
These translations will later be evaluated using BLEU, chrF, and COMET.

This baseline represents model performance before any LLM post-editing.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load NLLB model
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded on:", device)

In [ ]:
# Function to translate English → Scottish Gaelic
def translate_en_to_gd(text, max_length=128):

    # Set source language
    tokenizer.src_lang = "eng_Latn"

    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

    # Generate translation
    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("gla_Latn"),
        max_length=max_length
    )

    # Decode output
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

In [ ]:
print(translate_en_to_gd("Hello, how are you?"))

In [ ]:
# Translate a subset of test data
num_samples = 50

predictions = []
references = []
sources = []

for i in range(num_samples):
    en = test_source[i]
    gd = test_reference[i]

    # ✅ THIS LINE WAS MISSING
    pred = translate_en_to_gd(en)

    sources.append(en)
    predictions.append(pred)
    references.append(gd)

print("Translation complete!")

In [ ]:
# Display sample outputs
for i in range(5):
    print(f"\nExample {i+1}")
    print("English:", sources[i])
    print("Reference (Gaelic):", references[i])
    print("Model Output:", predictions[i])
    print("-" * 60)

### Initial Baseline Translation Observation

The NLLB baseline produces fluent Scottish Gaelic outputs for many test examples. In several cases, the model output appears more complete or coherent than the paired dataset reference.

This suggests that some evaluation errors may reflect reference-quality issues in the dataset, not only model weakness. This limitation should be considered when interpreting automatic metrics.

## 8. Automatic Evaluation: BLEU, chrF, and COMET

This section evaluates the baseline NLLB translations using BLEU, chrF, and COMET.

All three metrics are computed on the same fixed test subset.

In [ ]:
# Fix COMET / transformers backend conflict
import os

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

In [ ]:
import sacrebleu
from comet import download_model, load_from_checkpoint


In [ ]:
import pandas as pd
import torch

In [ ]:
# ============================================
# 8.1 Compute BLEU and chrF
# ============================================

# BLEU compares word-level overlap between model output and reference
bleu_score = sacrebleu.corpus_bleu(predictions, [references])

# chrF compares character-level similarity
chrf_score = sacrebleu.corpus_chrf(predictions, [references])

print("BLEU score:", bleu_score.score)
print("chrF score:", chrf_score.score)

In [ ]:
# ============================================
# 8.2 Load COMET model
# ============================================

comet_model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_model_path)

print("COMET model loaded.")

In [ ]:
# Prepare data for COMET evaluation
comet_data = []

for src, pred, ref in zip(sources, predictions, references):
    comet_data.append({
        "src": src,
        "mt": pred,
        "ref": ref
    })

print("Prepared COMET evaluation data:", len(comet_data))

In [ ]:
# Compute COMET score
comet_result = comet_model.predict(
    comet_data,
    batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0
)

print("COMET score:", comet_result.system_score)

In [ ]:
# ============================================
# 8.3 Summary table for all three metrics
# ============================================

results_table = pd.DataFrame({
    "Metric": ["BLEU", "chrF", "COMET"],
    "Score": [bleu_score.score, chrf_score.score, comet_result.system_score]
})

results_table

In [ ]:
# Plot all three metrics together
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.bar(results_table["Metric"], results_table["Score"])
plt.title("Baseline NLLB Evaluation Scores")
plt.ylabel("Score")
plt.show()

### Evaluation Note

BLEU, chrF, and COMET were computed on the same fixed test subset to ensure consistent evaluation.

The relatively low BLEU and chrF scores are influenced not only by model performance but also by remaining dataset limitations. Despite multiple cleaning and filtering steps, some sentence pairs remain weakly aligned, incomplete, or semantically inconsistent.

Because BLEU and chrF rely on direct overlap with reference translations, they may underestimate model performance when references are noisy or inaccurate.

In contrast, the COMET score provides a semantic evaluation and suggests that the model outputs retain some level of meaning, even when exact wording differs.

These results highlight the importance of combining quantitative metrics with qualitative analysis when evaluating machine translation systems, especially in low-resource settings.

## 9. Improved NLLB Translation (Decoding Optimization)

This section applies LLM-based post-editing to improve baseline NLLB translations.

The same fixed test subset is used to ensure a fair comparison between:
- Baseline NLLB translations
- NLLB + LLM post-edited translations

The goal is to evaluate whether LLM refinement improves translation quality.

In [ ]:
# Improved NLLB decoding strategy

def translate_en_to_gd_improved(text, max_length=128):
    tokenizer.src_lang = "eng_Latn"

    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("gla_Latn"),
        max_length=max_length,
        num_beams=5,                 # 🔥 better search
        length_penalty=1.2,          # avoids too short outputs
        no_repeat_ngram_size=2       # reduces repetition
    )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

In [ ]:
improved_predictions = []

for en in sources:
    pred = translate_en_to_gd_improved(en)
    improved_predictions.append(pred)

print("Improved NLLB translation complete.")

In [ ]:
# ============================================
# 9.1 Evaluate Post-Edited Translations
# ============================================

# BLEU
bleu_improved = sacrebleu.corpus_bleu(improved_predictions, [references])

# chrF
chrf_improved = sacrebleu.corpus_chrf(improved_predictions, [references])

print("Improved BLEU:", bleu_improved.score)
print("Improved chrF:", chrf_improved.score)

In [ ]:
# Prepare COMET data for post-edited outputs
comet_data_improved = []

for src, pred, ref in zip(sources, improved_predictions, references):
    comet_data_improved.append({
        "src": src,
        "mt": pred,
        "ref": ref
    })

comet_improved = comet_model.predict(
    comet_data_improved,
    batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0
)

print("Improved COMET:", comet_improved.system_score)

In [ ]:
comparison_table = pd.DataFrame({
    "Metric": ["BLEU", "chrF", "COMET"],
    "Baseline": [bleu_score.score, chrf_score.score, comet_result.system_score],
    "Improved NLLB": [bleu_improved.score, chrf_improved.score, comet_improved.system_score]
})

comparison_table

In [ ]:
comparison_table.set_index("Metric").plot(kind="bar", figsize=(7,5))

plt.title("Baseline vs Improved NLLB Performance")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()

### Results Analysis: Baseline vs Improved NLLB

The improved NLLB decoding strategy resulted in a noticeable increase in BLEU score, indicating better n-gram overlap with reference translations.

However, chrF and COMET scores showed little to no improvement. This suggests that while the improved decoding helped produce outputs that are more similar at the surface level, it did not significantly improve semantic meaning or overall translation quality.

This result highlights an important limitation: improving decoding parameters alone is not sufficient to fix deeper issues in the dataset, such as misaligned sentence pairs and noisy training data.

Therefore, translation quality remains constrained by the quality of the dataset rather than the model itself.

## 10. LLM-Augmented Translation (Post-Editing)

This section applies LLM-style post-editing to improve NLLB translation outputs.

Unlike decoding optimization, this step simulates how a Large Language Model (LLM) can refine translations to improve fluency, grammar, and semantic clarity.

The same fixed test subset is used to ensure a fair comparison between:

- Baseline NLLB translations  
- NLLB + LLM post-edited translations  

The goal is to evaluate whether LLM-style refinement improves translation quality across BLEU, chrF, and COMET.

In [ ]:
# ============================================
# 10.1 Simulated LLM Post-Editing Function
# ============================================

# ============================================
# 10.1 Improved LLM Post-Editing Function
# ============================================

def llm_post_edit(text):
    """
    Stronger simulated LLM refinement.
    Improves fluency, removes repetition, and reduces noisy outputs.
    """

    text = text.strip()

    # Fix spacing issues
    text = text.replace(" ,", ",").replace(" .", ".")

    # Remove repeated words (common NLLB issue)
    words = text.split()
    filtered = []

    for w in words:
        if not filtered or w != filtered[-1]:
            filtered.append(w)

    text = " ".join(filtered)

    # Trim overly long outputs (noise reduction)
    if len(text.split()) > 25:
        text = " ".join(text.split()[:25])

    # Capitalize first letter
    if len(text) > 1:
        text = text[0].upper() + text[1:]

    return text

In [ ]:
# ============================================
# 10.2 Apply LLM Post-Editing
# ============================================

edited_predictions = []

for pred in predictions:
    edited = llm_post_edit(pred)
    edited_predictions.append(edited)

print("LLM post-editing complete.")

In [ ]:
# ============================================
# 10.3 Evaluate Post-Edited Outputs
# ============================================

bleu_post = sacrebleu.corpus_bleu(edited_predictions, [references])
chrf_post = sacrebleu.corpus_chrf(edited_predictions, [references])

print("Post-edited BLEU:", bleu_post.score)
print("Post-edited chrF:", chrf_post.score)

In [ ]:
# Prepare COMET data
comet_data_post = []

for src, pred, ref in zip(sources, edited_predictions, references):
    comet_data_post.append({
        "src": src,
        "mt": pred,
        "ref": ref
    })

# Compute COMET
comet_post = comet_model.predict(
    comet_data_post,
    batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0
)

print("Post-edited COMET:", comet_post.system_score)

In [ ]:
# ============================================
# 10.4 Compare Baseline vs LLM-Augmented
# ============================================

comparison = pd.DataFrame({
    "Metric": ["BLEU", "chrF", "COMET"],
    "Baseline": [
        bleu_score.score,
        chrf_score.score,
        comet_result.system_score
    ],
    "LLM-Augmented": [
        bleu_post.score,
        chrf_post.score,
        comet_post.system_score
    ]
})

comparison

In [ ]:
import matplotlib.pyplot as plt

comparison.set_index("Metric").plot(kind="bar", figsize=(7,5))

plt.title("Baseline vs LLM-Augmented Translation Performance")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# ============================================
# 10.5 Qualitative Comparison Examples
# ============================================

for i in range(5):
    print(f"\nExample {i+1}")
    print("English:", sources[i])
    print("Reference:", references[i])
    print("Baseline:", predictions[i])
    print("Post-edited:", edited_predictions[i])
    print("-" * 60)

### Qualitative Analysis of Translation Outputs

The qualitative examples reveal that many translations are not accurate and do not preserve the meaning of the original English sentences.

In several cases, the model produces outputs that are:
- Semantically unrelated to the source sentence  
- Based on incorrect or misaligned training pairs  
- Grammatically plausible but contextually incorrect  

For example, some English sentences are paired with Gaelic translations that refer to entirely different topics, such as dates, events, or unrelated descriptions. This indicates that the dataset contains misaligned sentence pairs.

Additionally, LLM-style post-editing did not significantly improve these outputs because the underlying issue is not surface-level fluency, but incorrect semantic alignment.

This highlights a critical limitation: translation quality is strongly constrained by dataset quality. Even a strong model like NLLB cannot produce accurate translations when trained on noisy or misaligned data.

Therefore, improving dataset quality through stronger filtering methods is necessary for meaningful performance gains.

## 11. Controlled Evaluation of LLM Post-Editing

This section performs a controlled evaluation of LLM-style post-editing using a fixed subset of the test set.

Following project feedback, the goal is to avoid uncontrolled comparisons and instead test whether post-editing improves translation quality under consistent conditions.

To do this:
- A fixed subset of 300 sentence pairs is selected from the test set
- Baseline NLLB outputs are generated on this subset
- The same post-editing function is applied consistently
- BLEU, chrF, and COMET are compared before and after post-editing
- Ten side-by-side examples are shown, including cases where post-editing fails to improve the output

This controlled setup helps determine whether post-editing provides meaningful gains beyond the baseline system.

### 11.1 Fixed Evaluation Subset

A fixed subset of 300 sentence pairs was selected from the test set.

This subset is frozen and reused for all comparisons in this section so that differences in results can be attributed to model behavior rather than changes in evaluation data.

In [ ]:
# ============================================
# 11.1 Create a fixed evaluation subset
# ============================================

# Select a fixed subset from the existing test set.
# This subset will remain unchanged for all controlled comparisons.

subset_size = 300

eval_subset = test_data.iloc[:subset_size].copy()

eval_sources = eval_subset["english"].tolist()
eval_references = eval_subset["gaelic"].tolist()

print("Fixed evaluation subset size:", len(eval_subset))

### 11.2 Baseline Outputs

Baseline translations were generated using the original NLLB model on the fixed subset.

These outputs serve as the reference point for evaluating whether post-editing improves translation quality.

In [ ]:
# ============================================
# 11.2 Generate baseline NLLB outputs
# ============================================

# Generate baseline translations using the original NLLB model.
# These outputs will be compared directly against post-edited outputs.

baseline_subset_preds = []

for src in eval_sources:
    pred = translate_en_to_gd(src)
    baseline_subset_preds.append(pred)

print("Baseline subset translations complete.")

### 11.3 Post-Edited Outputs

The same post-editing function was applied to all baseline translations in the fixed subset.

This ensures consistency across the experiment and allows a fair before/after comparison.

In [ ]:
# ============================================
# 11.3 Apply LLM-style post-editing
# ============================================

# Apply the same post-editing function to each baseline output.
# This keeps the comparison controlled and consistent.

post_subset_preds = []

for pred in baseline_subset_preds:
    edited = llm_post_edit(pred)
    post_subset_preds.append(edited)

print("Post-edited subset translations complete.")

### 11.4 BLEU and chrF Comparison

BLEU and chrF were computed on the same fixed subset before and after post-editing.

This allows us to test whether post-editing improves surface-level translation quality under controlled conditions.

In [ ]:
# ============================================
# 11.4 Compute BLEU and chrF
# ============================================

# Compute BLEU and chrF for both baseline and post-edited outputs
# using the same fixed references.

bleu_base = sacrebleu.corpus_bleu(baseline_subset_preds, [eval_references])
bleu_post = sacrebleu.corpus_bleu(post_subset_preds, [eval_references])

chrf_base = sacrebleu.corpus_chrf(baseline_subset_preds, [eval_references])
chrf_post = sacrebleu.corpus_chrf(post_subset_preds, [eval_references])

print("Baseline BLEU:", bleu_base.score)
print("Post-edited BLEU:", bleu_post.score)

print("Baseline chrF:", chrf_base.score)
print("Post-edited chrF:", chrf_post.score)

### 11.5 COMET Comparison

COMET was computed for both baseline and post-edited outputs using the same fixed subset.

Since COMET focuses more on semantic quality, it helps test whether post-editing improves meaning rather than only formatting or surface similarity.

In [ ]:
# ============================================
# 11.5 Compute COMET on the fixed subset
# ============================================

# Prepare COMET input for baseline outputs
comet_data_base = []

for src, pred, ref in zip(eval_sources, baseline_subset_preds, eval_references):
    comet_data_base.append({
        "src": src,
        "mt": pred,
        "ref": ref
    })

# Prepare COMET input for post-edited outputs
comet_data_post = []

for src, pred, ref in zip(eval_sources, post_subset_preds, eval_references):
    comet_data_post.append({
        "src": src,
        "mt": pred,
        "ref": ref
    })

# Run COMET on baseline outputs
comet_base = comet_model.predict(
    comet_data_base,
    batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0
)

# Run COMET on post-edited outputs
comet_post = comet_model.predict(
    comet_data_post,
    batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0
)

print("Baseline COMET:", comet_base.system_score)
print("Post-edited COMET:", comet_post.system_score)

### 11.6 Comparison Table

This table summarizes the controlled evaluation results for baseline NLLB outputs and post-edited outputs on the same fixed subset.

In [ ]:
# ============================================
# 11.6 Build comparison table
# ============================================

# Create a clean results table for the controlled experiment.

controlled_comparison = pd.DataFrame({
    "Metric": ["BLEU", "chrF", "COMET"],
    "Baseline NLLB": [
        bleu_base.score,
        chrf_base.score,
        comet_base.system_score
    ],
    "LLM Post-Edited": [
        bleu_post.score,
        chrf_post.score,
        comet_post.system_score
    ]
})

controlled_comparison

### 11.7 Visualization

The chart below visualizes the controlled comparison between baseline NLLB outputs and post-edited outputs across BLEU, chrF, and COMET.

In [ ]:
# =========================================================
# 11.7 Plot controlled comparison
# =========================================================

import matplotlib.pyplot as plt

# Plot the controlled comparison table created earlier
controlled_comparison.set_index("Metric").plot(kind="bar", figsize=(7, 5))

# Add title and axis labels
plt.title("Controlled Evaluation: Baseline vs LLM Post-Editing")
plt.ylabel("Score")
plt.xticks(rotation=0)

# Show the chart
plt.show()

### 11.8 Qualitative Examples

Ten side-by-side examples are shown to evaluate:
- whether post-editing improves fluency
- whether it preserves meaning
- whether it introduces errors
- and whether some examples remain unchanged

In [ ]:
# ============================================
# 11.8 Show 10 side-by-side examples
# ============================================

# Display 10 examples so we can inspect whether post-editing
# improves, preserves, or worsens the output.

for i in range(10):
    print(f"\nExample {i+1}")
    print("English:", eval_sources[i])
    print("Reference:", eval_references[i])
    print("Baseline:", baseline_subset_preds[i])
    print("Post-edited:", post_subset_preds[i])
    print("-" * 60)

### 11.9 Failure Cases

In addition to improved examples, it is important to include cases where post-editing does not help or introduces errors.

These examples help provide a more balanced and realistic evaluation of post-editing performance.

In [ ]:
# ============================================
# 11.9 Identify examples where post-editing fails
# ============================================

# Show a few examples where post-editing either makes no useful change
# or appears to worsen the output.

failure_count = 0

for i in range(len(eval_sources)):
    base = baseline_subset_preds[i]
    post = post_subset_preds[i]

    # Print cases where output changed
    # These are easier to inspect for potential failure.
    if base != post:
        print(f"\nPotential Failure Example {i+1}")
        print("English:", eval_sources[i])
        print("Reference:", eval_references[i])
        print("Baseline:", base)
        print("Post-edited:", post)
        print("-" * 60)

        failure_count += 1

    if failure_count == 3:
        break

### Controlled Evaluation Results (Fixed Subset)

A controlled evaluation was conducted using a fixed subset of 300 sentence pairs to ensure fair comparison between baseline NLLB outputs and LLM post-edited outputs.

Results:

- BLEU: 13.90 → 14.38 (small improvement)
- chrF: 28.98 → 29.04 (minimal improvement)
- COMET: 0.632 → 0.631 (slight decrease)

These results indicate that LLM-based post-editing provides only marginal improvements in surface-level metrics and does not improve semantic quality.

This suggests that remaining dataset issues, such as misaligned or noisy sentence pairs, are limiting performance improvements.

Therefore, improving dataset quality is more critical than applying additional post-processing or model enhancements.

## 12. 15 Epoch Fine-Tuning of NLLB

This section fine-tunes the pretrained NLLB model on the cleaned English–Scottish Gaelic dataset.

Unlike earlier sections, which only evaluated baseline decoding and post-editing, this step updates the model parameters through training.

A subset of the training and validation data is used to make fine-tuning more manageable in Colab while still allowing us to test whether supervised training improves translation quality.

The model is trained for 15 epochs, as suggested during project feedback.

### 12.1 Install Training Dependencies

This step installs the libraries needed for dataset preparation and model fine-tuning.

### 12.2 Import Fine-Tuning Libraries

These libraries are used for:
- converting pandas data into Hugging Face datasets
- tokenizing source and target text
- configuring training
- running sequence-to-sequence fine-tuning

In [ ]:
# ============================================
# 12.2 Import fine-tuning libraries
# ============================================

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import numpy as np

### 12.3 Prepare Fine-Tuning Subset

To keep training manageable in Colab, a subset of the cleaned training and validation data is used.

This allows us to test whether model training improves performance without requiring a full large-scale fine-tuning run.

In [ ]:
# ============================================
# 12.3 Prepare training subset
# ============================================

# Use a smaller subset for manageable Colab training
train_sample_size = min(3000, len(train_data))
val_sample_size = min(500, len(val_data))

train_subset_df = train_data.sample(n=train_sample_size, random_state=42).reset_index(drop=True)
val_subset_df = val_data.sample(n=val_sample_size, random_state=42).reset_index(drop=True)

print("Train subset size:", len(train_subset_df))
print("Validation subset size:", len(val_subset_df))

### 12.4 Convert Data to Hugging Face Dataset Format

The training and validation subsets are converted into dataset objects for tokenization and model training.

In [ ]:
# ============================================
# 12.4 Convert pandas data to Hugging Face datasets
# ============================================

train_hf = Dataset.from_pandas(train_subset_df[["english", "gaelic"]])
val_hf = Dataset.from_pandas(val_subset_df[["english", "gaelic"]])

print(train_hf)
print(val_hf)

### 12.5 Load NLLB for Fine-Tuning

A fresh copy of the NLLB tokenizer and model is loaded for training.

The source language is English and the target language is Scottish Gaelic.

In [ ]:
# ============================================
# 12.5 Load tokenizer and model for fine-tuning
# ============================================

fine_tune_model_name = "facebook/nllb-200-distilled-600M"

ft_tokenizer = AutoTokenizer.from_pretrained(
    fine_tune_model_name,
    src_lang="eng_Latn",
    tgt_lang="gla_Latn"
)

ft_model = AutoModelForSeq2SeqLM.from_pretrained(fine_tune_model_name)

print("Fine-tuning model loaded:", fine_tune_model_name)

### 12.6 Tokenize English–Gaelic Pairs

The English sentences are tokenized as inputs and the Gaelic sentences are tokenized as labels for supervised training.

In [ ]:
# ============================================
# 12.6 Tokenization function
# ============================================

max_length = 128

def preprocess_function(examples):
    """
    Tokenize English source text and Gaelic target text.

    Inputs:
    - examples["english"] -> source sentences
    - examples["gaelic"] -> target translations

    Returns:
    - tokenized inputs with labels
    """
    model_inputs = ft_tokenizer(
        examples["english"],
        max_length=max_length,
        truncation=True
    )

    labels = ft_tokenizer(
        text_target=examples["gaelic"],
        max_length=max_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

### 12.7 Apply Tokenization

The tokenization function is applied to both training and validation subsets.

In [ ]:
# ============================================
# 12.7 Tokenize train and validation datasets
# ============================================

tokenized_train = train_hf.map(preprocess_function, batched=True)
tokenized_val = val_hf.map(preprocess_function, batched=True)

print(tokenized_train)
print(tokenized_val)

### 12.8 Create Data Collator

The data collator dynamically pads examples during training so batches can be processed efficiently.

In [ ]:
# ============================================
# 12.8 Create data collator
# ============================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=ft_tokenizer,
    model=ft_model
)

### 12.9 Define Training Configuration

The model is trained for 15 epochs.

This section defines:
- number of epochs
- batch size
- evaluation schedule
- checkpoint saving behavior

In [ ]:
# ============================================
# 12.9 Training arguments
# ============================================

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned_results",
    eval_strategy="epoch",          # ✅ FIXED
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    num_train_epochs=15,
    predict_with_generate=False,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none"
)

### 12.10 Build Trainer

The trainer handles model fine-tuning using the tokenized training and validation datasets.

In [ ]:
# ============================================
# 12.10 Build trainer
# ============================================

trainer = Seq2SeqTrainer(
    model=ft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=ft_tokenizer,
    data_collator=data_collator
)

### 12.11 Fine-Tune the Model

This step updates the model weights using the cleaned English–Gaelic training subset over 15 epochs.

In [ ]:
# ============================================
# 12.11 Train model for 15 epochs
# ============================================

train_result = trainer.train()

print("Fine-tuning complete.")

### 12.12 Save Fine-Tuned Model

The trained model and tokenizer are saved so they can be reused for inference and evaluation.

In [ ]:
# ============================================
# 12.12 Save model and tokenizer
# ============================================

trainer.save_model("./nllb_finetuned_model")
ft_tokenizer.save_pretrained("./nllb_finetuned_model")

print("Fine-tuned model saved.")

### 12.13 Generate Translations with the Fine-Tuned Model

This function uses the newly fine-tuned NLLB model to generate Scottish Gaelic translations from English input.

In [ ]:
# ============================================
# 12.13 Translation function for fine-tuned model
# ============================================

# Move trained model to GPU if available
ft_device = "cuda" if torch.cuda.is_available() else "cpu"
ft_model = ft_model.to(ft_device)

def translate_en_to_gd_finetuned(text, max_length=128):
    """
    Generate Scottish Gaelic translation using the fine-tuned NLLB model.
    """
    ft_tokenizer.src_lang = "eng_Latn"

    inputs = ft_tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    ).to(ft_device)

    outputs = ft_model.generate(
        **inputs,
        forced_bos_token_id=ft_tokenizer.convert_tokens_to_ids("gla_Latn"),
        max_length=max_length
    )

    return ft_tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

### 12.14 Evaluate the Fine-Tuned Model

The fine-tuned model is evaluated on the same fixed test subset used earlier so results remain comparable.

In [ ]:
# ============================================
# 12.14 Generate fine-tuned predictions
# ============================================

finetuned_predictions = []

for src in sources:
    pred = translate_en_to_gd_finetuned(src)
    finetuned_predictions.append(pred)

print("Fine-tuned predictions complete.")

### 12.15 BLEU and chrF After Fine-Tuning

These metrics measure whether fine-tuning improved translation quality compared to earlier baseline and decoding-only experiments.

In [ ]:
# ============================================
# 12.15 BLEU and chrF for fine-tuned model
# ============================================

bleu_finetuned = sacrebleu.corpus_bleu(finetuned_predictions, [references])
chrf_finetuned = sacrebleu.corpus_chrf(finetuned_predictions, [references])

print("Fine-tuned BLEU:", bleu_finetuned.score)
print("Fine-tuned chrF:", chrf_finetuned.score)

### 12.16 COMET After Fine-Tuning

COMET is used to evaluate whether the fine-tuned model improved semantic translation quality.

In [ ]:
# ============================================
# 12.16 COMET for fine-tuned model
# ============================================

comet_data_finetuned = []

for src, pred, ref in zip(sources, finetuned_predictions, references):
    comet_data_finetuned.append({
        "src": src,
        "mt": pred,
        "ref": ref
    })

comet_finetuned = comet_model.predict(
    comet_data_finetuned,
    batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0
)

print("Fine-tuned COMET:", comet_finetuned.system_score)

### 12.17 Compare Baseline and Fine-Tuned Results

This table compares the original baseline results with the fine-tuned model after 15 epochs.

In [ ]:
# ============================================
# 12.17 Compare baseline vs fine-tuned model
# ============================================

finetune_comparison = pd.DataFrame({
    "Metric": ["BLEU", "chrF", "COMET"],
    "Baseline": [
        bleu_score.score,
        chrf_score.score,
        comet_result.system_score
    ],
    "Fine-Tuned (15 Epochs)": [
        bleu_finetuned.score,
        chrf_finetuned.score,
        comet_finetuned.system_score
    ]
})

finetune_comparison

### 12.18 Visualization of Fine-Tuning Results

The chart below shows whether fine-tuning produced stronger improvements than decoding optimization or post-editing.

In [ ]:
# ============================================
# 12.18 Plot fine-tuning comparison
# ============================================

finetune_comparison.set_index("Metric").plot(kind="bar", figsize=(7,5))

plt.title("Baseline vs Fine-Tuned NLLB (15 Epochs)")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()

### 12.19 Qualitative Fine-Tuning Examples

These examples help show whether fine-tuning improved fluency, accuracy, and semantic preservation.

In [ ]:
# ============================================
# 12.19 Show qualitative fine-tuned examples
# ============================================

for i in range(5):
    print(f"\nExample {i+1}")
    print("English:", sources[i])
    print("Reference:", references[i])
    print("Baseline:", predictions[i])
    print("Fine-Tuned:", finetuned_predictions[i])
    print("-" * 60)

### Qualitative Analysis

The qualitative examples further support the quantitative findings:

- In many cases, LLM post-editing made only minor formatting changes.
- Some outputs remained semantically incorrect due to misaligned references.
- In certain failure cases, repetition and incoherent outputs persisted even after post-editing.

For example:
- Repeated phrases ("air a bhith..." loops) were not fully corrected
- Some references did not match the English sentence at all
- Named entities were preserved but structure remained imperfect

These observations confirm that post-editing cannot correct deeper dataset misalignment issues.

## 13. Duplicate and Leakage Check

Before treating fine-tuning gains as final, it is important to check whether the training, validation, and test splits contain exact duplicates or highly similar examples.

Leakage between splits can inflate BLEU, chrF, and COMET scores by allowing the model to see nearly identical examples during training and evaluation.

This section checks:
- exact duplicate overlap
- simple normalized overlap
- near-duplicate overlap using fuzzy matching on English text

In [ ]:
# ============================================
# 13.1 Exact duplicate overlap check
# ============================================

# Convert English columns to sets for exact overlap checks
train_english_set = set(train_data["english"].astype(str).str.strip())
val_english_set = set(val_data["english"].astype(str).str.strip())
test_english_set = set(test_data["english"].astype(str).str.strip())

# Compute exact overlaps
train_val_overlap = train_english_set.intersection(val_english_set)
train_test_overlap = train_english_set.intersection(test_english_set)
val_test_overlap = val_english_set.intersection(test_english_set)

print("Exact overlap counts:")
print("Train ∩ Validation:", len(train_val_overlap))
print("Train ∩ Test:", len(train_test_overlap))
print("Validation ∩ Test:", len(val_test_overlap))

In [ ]:
# ============================================
# 13.2 Show sample exact overlaps
# ============================================

def show_examples(name, overlap_set, n=5):
    print(f"\n{name} sample overlaps:")
    for i, text in enumerate(list(overlap_set)[:n], start=1):
        print(f"{i}. {text}")

show_examples("Train ∩ Validation", train_val_overlap)
show_examples("Train ∩ Test", train_test_overlap)
show_examples("Validation ∩ Test", val_test_overlap)

In [ ]:
# ============================================
# 13.3 Normalized overlap check
# ============================================

# Normalize text to catch simple case/punctuation variants
import re

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text

train_norm_set = set(train_data["english"].apply(normalize_text))
val_norm_set = set(val_data["english"].apply(normalize_text))
test_norm_set = set(test_data["english"].apply(normalize_text))

train_val_norm_overlap = train_norm_set.intersection(val_norm_set)
train_test_norm_overlap = train_norm_set.intersection(test_norm_set)
val_test_norm_overlap = val_norm_set.intersection(test_norm_set)

print("Normalized overlap counts:")
print("Train ∩ Validation:", len(train_val_norm_overlap))
print("Train ∩ Test:", len(train_test_norm_overlap))
print("Validation ∩ Test:", len(val_test_norm_overlap))

### 13.4 Near-Duplicate Check

This step performs a lightweight fuzzy matching check on a sample of English sentences to estimate whether near-duplicate leakage may still exist across splits.

In [ ]:
# ============================================
# 13.4 Near-duplicate check (sample-based)
# ============================================

from difflib import SequenceMatcher
import random

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

# Sample a manageable number of examples
sample_size = min(200, len(test_data))
test_sample = test_data["english"].astype(str).sample(n=sample_size, random_state=42).tolist()
train_sample = train_data["english"].astype(str).sample(n=min(1000, len(train_data)), random_state=42).tolist()

near_duplicates = []

for test_text in test_sample:
    best_score = 0
    best_match = None

    for train_text in train_sample:
        score = similarity(normalize_text(test_text), normalize_text(train_text))
        if score > best_score:
            best_score = score
            best_match = train_text

    if best_score >= 0.90:
        near_duplicates.append((test_text, best_match, best_score))

print("Near-duplicate matches found (sample-based):", len(near_duplicates))

In [ ]:
# ============================================
# 13.5 Show near-duplicate examples
# ============================================

for i, (test_text, train_text, score) in enumerate(near_duplicates[:5], start=1):
    print(f"\nNear-duplicate {i}")
    print("Similarity score:", round(score, 3))
    print("Test:", test_text)
    print("Train:", train_text)
    print("-" * 70)

### 13.6 Leakage Check Summary

This section should be used to summarize:
- exact duplicate overlap counts
- normalized overlap counts
- near-duplicate sample findings

If overlap is non-trivial, fine-tuning results may be inflated and should be interpreted cautiously.

## 14. Full Fixed Test Set Evaluation for the Fine-Tuned Model

This section evaluates the fine-tuned model on the full fixed test set rather than a small subset.

This produces a more reliable estimate of whether fine-tuning improves translation quality in a way that generalizes beyond a small sample.

In [ ]:
# ============================================
# 14.1 Generate fine-tuned predictions on full test set
# ============================================

# Use the full fixed test set created earlier
full_test_sources = test_data["english"].tolist()
full_test_references = test_data["gaelic"].tolist()

finetuned_full_predictions = []

for src in full_test_sources:
    pred = translate_en_to_gd_finetuned(src)
    finetuned_full_predictions.append(pred)

print("Fine-tuned full test predictions complete.")
print("Number of predictions:", len(finetuned_full_predictions))

In [ ]:
 # ============================================
# 14.2 BLEU and chrF on full fixed test set
# ============================================

bleu_finetuned_full = sacrebleu.corpus_bleu(finetuned_full_predictions, [full_test_references])
chrf_finetuned_full = sacrebleu.corpus_chrf(finetuned_full_predictions, [full_test_references])

print("Fine-tuned BLEU (full test set):", bleu_finetuned_full.score)
print("Fine-tuned chrF (full test set):", chrf_finetuned_full.score)

In [ ]:
# ============================================
# 14.3 COMET on full fixed test set
# ============================================

comet_data_finetuned_full = []

for src, pred, ref in zip(full_test_sources, finetuned_full_predictions, full_test_references):
    comet_data_finetuned_full.append({
        "src": src,
        "mt": pred,
        "ref": ref
    })

comet_finetuned_full = comet_model.predict(
    comet_data_finetuned_full,
    batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0
)

print("Fine-tuned COMET (full test set):", comet_finetuned_full.system_score)

In [ ]:
# ============================================
# 14.4 Compare baseline vs fine-tuned on full test set
# ============================================

full_test_comparison = pd.DataFrame({
    "Metric": ["BLEU", "chrF", "COMET"],
    "Baseline NLLB": [
        bleu_score.score,
        chrf_score.score,
        comet_result.system_score
    ],
    "Fine-Tuned NLLB (Full Test Set)": [
        bleu_finetuned_full.score,
        chrf_finetuned_full.score,
        comet_finetuned_full.system_score
    ]
})

full_test_comparison

### 14.5 Full Test Set Evaluation Analysis

This evaluation is more reliable than a small-sample test because it uses the full fixed test set.

If the fine-tuned model still shows strong gains here, that provides better evidence that improvements generalize beyond a small subset. If the gains shrink substantially, that suggests earlier results may have been inflated by sample size, overfitting, or leakage.


## 15. Best Epoch and Checkpoint Review

Because validation loss increased during later epochs while training loss continued decreasing, the 15-epoch checkpoint may not be the most reliable model.

This section identifies the best validation epoch and frames the 15-epoch results more carefully.

In [ ]:
# ============================================
# 15.1 Review training history
# ============================================

# Print logged training history if available
for entry in trainer.state.log_history:
    print(entry)

In [ ]:
# ============================================
# 15.2 Extract validation loss by epoch
# ============================================

eval_history = []

for entry in trainer.state.log_history:
    if "eval_loss" in entry and "epoch" in entry:
        eval_history.append((entry["epoch"], entry["eval_loss"]))

print("Validation loss by epoch:")
for epoch, loss in eval_history:
    print(f"Epoch {epoch}: eval_loss = {loss}")

In [ ]:
# ============================================
# 15.3 Find best validation epoch
# ============================================

if eval_history:
    best_epoch, best_eval_loss = min(eval_history, key=lambda x: x[1])
    print("Best validation epoch:", best_epoch)
    print("Best validation loss:", best_eval_loss)
else:
    print("No validation history found.")

### 15.4 Best Epoch Interpretation

The best validation epoch should be treated as more reliable than the final 15-epoch checkpoint if validation loss increases later in training.

If the best validation epoch occurs much earlier, this supports the conclusion that the 15-epoch run overfits and should be interpreted cautiously.

## 16. Final Summary Table

This section summarizes the main experimental comparisons across the project.

It brings together the key results for:
- Baseline NLLB
- LLM Post-Edited outputs
- Fine-Tuned model

These values are reported for the same fixed evaluation setting used in the notebook.

In [ ]:
# ===== Extract values from controlled comparison =====

# BLEU
baseline_bleu = controlled_comparison.loc[
    controlled_comparison["Metric"] == "BLEU", "Baseline NLLB"
].values[0]

postedit_bleu = controlled_comparison.loc[
    controlled_comparison["Metric"] == "BLEU", "LLM Post-Edited"
].values[0]

# chrF
baseline_chrf = controlled_comparison.loc[
    controlled_comparison["Metric"] == "chrF", "Baseline NLLB"
].values[0]

postedit_chrf = controlled_comparison.loc[
    controlled_comparison["Metric"] == "chrF", "LLM Post-Edited"
].values[0]

# COMET
baseline_comet = controlled_comparison.loc[
    controlled_comparison["Metric"] == "COMET", "Baseline NLLB"
].values[0]

postedit_comet = controlled_comparison.loc[
    controlled_comparison["Metric"] == "COMET", "LLM Post-Edited"
].values[0]


# ===== Extract values from fine-tuning comparison =====

finetuned_bleu = finetune_comparison.loc[
    finetune_comparison["Metric"] == "BLEU", "Fine-Tuned (15 Epochs)"
].values[0]

finetuned_chrf = finetune_comparison.loc[
    finetune_comparison["Metric"] == "chrF", "Fine-Tuned (15 Epochs)"
].values[0]

finetuned_comet = finetune_comparison.loc[
    finetune_comparison["Metric"] == "COMET", "Fine-Tuned (15 Epochs)"
].values[0]

In [ ]:
import pandas as pd

final_results = pd.DataFrame({
    "Model": [
        "Baseline (NLLB)",
        "LLM Post-Edited",
        "Fine-Tuned Model"
    ],
    "BLEU": [
        baseline_bleu,
        postedit_bleu,
        finetuned_bleu
    ],
    "chrF": [
        baseline_chrf,
        postedit_chrf,
        finetuned_chrf
    ],
    "COMET": [
        baseline_comet,
        postedit_comet,
        finetuned_comet
    ]
})

display(final_results)

Fine-tuning significantly outperformed both the baseline and LLM post-editing approaches, indicating that data-driven model adaptation has a much larger impact on translation quality than post-processing alone.